## Part D — Linear Learner Baseline
Goal: train SageMaker Linear Learner on the SAME processed data from Part A, so we can compare it
head-to-head against XGBoost. This is a baseline comparison to test whether XGBoost's extra
complexity was justified. Setup mirrors Part B — same session, role, region, and S3 paths.

In [1]:
import sagemaker, boto3
from sagemaker import get_execution_role

sess   = sagemaker.Session()
region = sess.boto_region_name
role   = get_execution_role()

bucket = "week6-churn-ah"   # same bucket as Parts A/B
prefix = "churn"

print("region:", region)
print("role:  ", role)
print("bucket:", bucket)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


region: eu-north-1
role:   arn:aws:iam::481719141347:role/service-role/AmazonSageMaker-ExecutionRole-20260812T005875
bucket: week6-churn-ah


## Reuse the Part A Processed Data
Key point: Linear Learner needs SCALED features, and our Part A data is already median-imputed,
scaled, and one-hot encoded. So we point at the exact same S3 files XGBoost used — no new
preprocessing. (This is the payoff of scaling in Part A even though XGBoost didn't need it.)

In [2]:
from sagemaker.inputs import TrainingInput

train_s3     = f"s3://{bucket}/data/train.csv"
val_s3       = f"s3://{bucket}/data/validation.csv"
test_feat_s3 = f"s3://{bucket}/data/test_features.csv"
test_s3      = f"s3://{bucket}/data/test.csv"          # labeled, for evaluation later

# same label-first, headerless CSV format from Part A
train_input = TrainingInput(train_s3, content_type="text/csv")
val_input   = TrainingInput(val_s3,   content_type="text/csv")

print("train:     ", train_s3)
print("validation:", val_s3)
print("test feats:", test_feat_s3)

train:      s3://week6-churn-ah/data/train.csv
validation: s3://week6-churn-ah/data/validation.csv
test feats: s3://week6-churn-ah/data/test_features.csv


## Linear Learner Estimator
Fetch SageMaker's built-in Linear Learner container (instead of XGBoost) and define the training job.
Same hardware and role as Part B — only the algorithm changes. Output goes to a SEPARATE S3 folder
so it never clashes with the XGBoost artifacts.

In [3]:
# 1) fetch the managed Linear Learner container image for this region
ll_image = sagemaker.image_uris.retrieve("linear-learner", region)
print("container:", ll_image)

# 2) define the training job (same hardware as XGBoost, separate output path)
ll = sagemaker.estimator.Estimator(
    image_uri         = ll_image,
    role              = role,
    instance_count    = 1,
    instance_type     = "ml.m5.large",
    output_path       = f"s3://{bucket}/{prefix}/ll-output",   # separate from XGBoost's output
    max_run           = 3600,
    sagemaker_session = sess,
)

print("Linear Learner estimator created")

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: Estimator is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
Estimator is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


container: 669576153137.dkr.ecr.eu-north-1.amazonaws.com/linear-learner:1
Linear Learner estimator created


## Hyperparameters — note the imbalance handling differs from XGBoost
Linear Learner has NO scale_pos_weight. Its equivalent is positive_example_weight_mult="balanced",
which auto-weights the rare churn class. We also tell it to pick its best internal model by F1
(imbalance-appropriate), and set feature_dim=30 (our processed feature count).

In [4]:
ll.set_hyperparameters(
    predictor_type                            = "binary_classifier",  # yes/no churn
    feature_dim                               = 30,                    # 30 processed features (from Part A)
    positive_example_weight_mult              = "balanced",            # <-- the imbalance knob (replaces scale_pos_weight)
    binary_classifier_model_selection_criteria= "f1",                  # pick best internal model by F1, not accuracy
    mini_batch_size                           = 200,
    epochs                                    = 15,
)

print("hyperparameters set — ready to train")

hyperparameters set — ready to train


## Train Linear Learner (single job — this is a baseline)
One training job on train + validation. No HPO here: a baseline's job is to answer "how well does a
simple linear model do out of the box?", so a single well-configured run is the fair, cheap comparison
against tuned

In [5]:
ll.fit({"train": train_input, "validation": val_input})

print("\ntraining complete")
print("model artifact:", ll.model_data)

INFO:sagemaker:Creating training-job with name: linear-learner-2026-08-16-09-52-51-879


2026-08-16 09:52:52 Starting - Starting the training job...
2026-08-16 09:53:13 Starting - Preparing the instances for training...
2026-08-16 09:53:30 Downloading - Downloading input data...
2026-08-16 09:54:10 Downloading - Downloading the training image......
2026-08-16 09:55:26 Training - Training image download completed. Training in progress...Docker entrypoint called with argument(s): train
Running default environment configuration script
[08/16/2026 09:55:30 INFO 139661173180224] Reading default configuration from /opt/amazon/lib/python3.8/site-packages/algorithm/resources/default-input.json: {'mini_batch_size': '1000', 'epochs': '15', 'feature_dim': 'auto', 'use_bias': 'true', 'binary_classifier_model_selection_criteria': 'accuracy', 'f_beta': '1.0', 'target_recall': '0.8', 'target_precision': '0.8', 'num_models': 'auto', 'num_calibration_samples': '10000000', 'init_method': 'uniform', 'init_scale': '0.07', 'init_sigma': '0.01', 'init_bias': '0.0', 'optimizer': 'auto', 'loss': 

## Batch Transform — score the SAME unseen customers
Score test_features.csv (the same 845 customers XGBoost scored). Linear Learner's batch output
returns predicted LABELS (its score field comes back as NaN in batch mode — a known quirk), so we
use these labels for test-set precision/recall/F1, and take AUC from the training-log validation score.

In [13]:
ll_transformer = ll.transformer(
    instance_count = 1,
    instance_type  = "ml.m5.large",
    output_path    = f"s3://{bucket}/{prefix}/ll-batch-final",
    accept         = "application/json",
    assemble_with  = "Line",
)

ll_transformer.transform(
    data         = test_feat_s3,
    content_type = "text/csv",
    split_type   = "Line",
)
ll_transformer.wait()
print("predictions written to:", ll_transformer.output_path)

INFO:sagemaker:Creating model with name: linear-learner-2026-08-16-11-23-12-411
INFO:sagemaker:Creating transform job with name: linear-learner-2026-08-16-11-23-12-960


................................Docker entrypoint called with argument(s): serve
Running default environment configuration script
[08/16/2026 11:28:31 INFO 139982901987136] Memory profiler is not enabled by the environment variable ENABLE_PROFILER.
/opt/amazon/lib/python3.8/site-packages/mxnet/model.py:97: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if num_device is 1 and 'dist' not in kvstore:
/opt/amazon/lib/python3.8/site-packages/scipy/optimize/_shgo.py:495: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if cons['type'] is 'ineq':
/opt/amazon/lib/python3.8/site-packages/scipy/optimize/_shgo.py:743: SyntaxWarning: "is not" with a literal. Did you mean "!="?
  if len(self.X_min) is not 0:
[08/16/2026 11:28:35 WARNING 139982901987136] Loggers have already been setup.
[08/16/2026 11:28:35 INFO 139982901987136] loaded entry point class algorithm.serve.server_config:config_api
[08/16/2026 11:28:35 INFO 139982901987136] loading entry points
[08/16/2026 11:28:35 INFO

## Load LL Predictions + True Labels
Parse the predicted labels (0/1) from the batch output and line them up against the true labels in
test.csv. We'll compute precision/recall/F1 from these; AUC comes from the validation log.

In [18]:
import pandas as pd, numpy as np, boto3, io, json
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

s3 = boto3.client("s3")
key = f"{prefix}/ll-batch-final/test_features.csv.out"
body = s3.get_object(Bucket=bucket, Key=key)["Body"].read().decode("utf-8")

# Linear Learner batch output can be one big JSON object OR one record per line — handle both
body = body.strip()
records = []
try:
    obj = json.loads(body)                      # case 1: whole file is one JSON object
    records = obj["predictions"] if isinstance(obj, dict) else obj
except json.JSONDecodeError:
    for line in body.split("\n"):               # case 2: one JSON record per line
        records.append(json.loads(line))

ll_pred = np.array([int(r["predicted_label"]) if isinstance(r, dict) else int(r)
                    for r in records])
print("parsed labels:", ll_pred.shape)          # expect (845,)

# true labels from test.csv column 0
obj = s3.get_object(Bucket=bucket, Key="data/test.csv")["Body"].read()
y_true = pd.read_csv(io.BytesIO(obj), header=None)[0].values
assert len(ll_pred) == len(y_true), f"row mismatch! preds={len(ll_pred)} truth={len(y_true)}"

print("\nLINEAR LEARNER — test set (from labels):")
print(f"  Precision: {precision_score(y_true, ll_pred):.4f}")
print(f"  Recall:    {recall_score(y_true, ll_pred):.4f}")
print(f"  F1:        {f1_score(y_true, ll_pred):.4f}")
tn, fp, fn, tp = confusion_matrix(y_true, ll_pred).ravel()
print(f"  Confusion: TN={tn} FP={fp} FN={fn} TP={tp}")
print("  (ROC-AUC from validation log = 0.8922; batch scores were NaN)")

parsed labels: (845,)

LINEAR LEARNER — test set (from labels):
  Precision: 0.4917
  Recall:    0.8310
  F1:        0.6178
  Confusion: TN=581 FP=122 FN=24 TP=118
  (ROC-AUC from validation log = 0.8922; batch scores were NaN)


In [11]:
import pandas as pd

ll_val  = {"ROC-AUC": 0.8922, "Precision": 0.5374, "Recall": 0.8099, "F1": 0.6461}
xgb_val = {"ROC-AUC": 0.9913, "Precision": 0.9577, "Recall": 0.9577, "F1": 0.9577}

comparison = pd.DataFrame({"XGBoost": xgb_val, "Linear Learner": ll_val})
comparison["XGBoost wins by"] = comparison["XGBoost"] - comparison["Linear Learner"]
print("MODEL COMPARISON — validation set, same data & metrics\n")
print(comparison.round(4).to_string())
print("\nNote: both rows are validation-set scores (fair, like-for-like).")
print("XGBoost was additionally confirmed on the untouched test set (AUC 0.9976).")

MODEL COMPARISON — validation set, same data & metrics

           XGBoost  Linear Learner  XGBoost wins by
ROC-AUC     0.9913          0.8922           0.0991
Precision   0.9577          0.5374           0.4203
Recall      0.9577          0.8099           0.1478
F1          0.9577          0.6461           0.3116

Note: both rows are validation-set scores (fair, like-for-like).
XGBoost was additionally confirmed on the untouched test set (AUC 0.9976).


## Conclusion
XGBoost (~0.99 AUC) decisively beats the Linear Learner baseline (~0.89) on the same data and metrics.
Linear Learner reaches good recall but low precision (~0.54) — it over-flags — which is textbook for a
linear boundary on a non-linear problem. This confirms churn here depends on non-linear patterns and
feature interactions, so XGBoost's added complexity is justified. I validated the model choice by
measuring it against a simple baseline rather than assuming it.

In [17]:
!pip install -U scikit-learn


  Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (9.7 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.7 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]
